In [1]:
#import libraries

import pandas as pd
from collections import Counter
from tqdm import tqdm

In [2]:
#load extracted entities

entity_df=pd.read_csv(
    r"C:\Users\arsha\OneDrive\Desktop\Healthcare Retrieval using qql\Data\extracted_entities.csv"
)

print(entity_df.shape)

display(entity_df.head())

(9276, 5)


,row_id,original_text,entity_text,entity_label,confidence
0,0,\n Patient Query:\n I woke up this morni...,panadol,medication,0.6307
1,0,\n Patient Query:\n I woke up this morni...,benign paroxysmal positional vertigo,medical_condition,0.7313
2,0,\n Patient Query:\n I woke up this morni...,BPPV,medical_condition,0.5871
3,0,\n Patient Query:\n I woke up this morni...,peripheral vertigo,medical_condition,0.5706
4,0,\n Patient Query:\n I woke up this morni...,dizziness,symptom,0.8374


In [3]:
#check entity labels

print(
    entity_df["entity_label"]
    .value_counts()
)

entity_label
body_part            2440
symptom              1915
medication           1656
medical_condition    1210
disease              1185
treatment             870
Name: count, dtype: int64


In [4]:
#relationship storage

relationships=[]

In [5]:
#group by healthcare conversation

grouped_data=entity_df.groupby(
    "row_id"
)

In [7]:
#create filtered relationships

for row_id,group in tqdm(grouped_data):

    symptoms=group[
        group["entity_label"]=="symptom"
    ]["entity_text"].str.lower().unique()

    diseases=group[
        group["entity_label"]=="disease"
    ]["entity_text"].str.lower().unique()

    conditions=group[
        group["entity_label"]=="medical_condition"
    ]["entity_text"].str.lower().unique()

    medications=group[
        group["entity_label"]=="medication"
    ]["entity_text"].str.lower().unique()

    #symptom -> disease
    for symptom in symptoms:
        for disease in diseases:
            relationships.append({
                "source":symptom,
                "target":disease,
                "relationship":"RELATED_TO"
            })
    #symptom -> condition
    for symptom in symptoms:
        for condition in conditions:
            relationships.append({
                "source":symptom,
                "target":condition,
                "relationship":"RELATED_TO"
            })
    #condition -> medication
    for condition in conditions:
        for medication in medications:
            relationships.append({
                "source":condition,
                "target":medication,
                "relationship":"TREATED_BY"

            })

100%|███████████████████████████████████████████████████████████████████████████████| 993/993 [00:03<00:00, 291.49it/s]


In [8]:
#create relationship dataframe

relationship_df=pd.DataFrame(
    relationships
)

print(relationship_df.shape)

display(relationship_df.head())

(7300, 3)


,source,target,relationship
0,dizziness,benign paroxysmal positional vertigo,RELATED_TO
1,dizziness,bppv,RELATED_TO
2,dizziness,peripheral vertigo,RELATED_TO
3,giddiness,benign paroxysmal positional vertigo,RELATED_TO
4,giddiness,bppv,RELATED_TO


In [9]:
#count relationship frequency

relationship_counts=relationship_df.groupby(
    ["source","target","relationship"]
).size().reset_index(name="count")

print(relationship_counts.shape)

display(
    relationship_counts.head()
)

(3483, 4)


,source,target,relationship,count
0,abcess,antibiotics,TREATED_BY,2
1,abcess,painkillers,TREATED_BY,2
2,abdominal cramping,miscarriage,RELATED_TO,2
3,abdominal cramping,miscarriages,RELATED_TO,2
4,abdominal cramping,period,RELATED_TO,2


In [10]:
#filter weak relationships

filtered_relationships=relationship_counts[
    relationship_counts["count"]>=5
]

print(filtered_relationships.shape)

display(
    filtered_relationships.head(20)
)

(24, 4)


,source,target,relationship,count
614,cough,asthma,RELATED_TO,10
622,cough,bronchitis,RELATED_TO,14
647,cough,sinusitis,RELATED_TO,6
1007,fever,infection,RELATED_TO,6
1031,fever,paracetamol,TREATED_BY,6
1413,infection,antibiotics,TREATED_BY,10
1422,infection,ibuprofen,TREATED_BY,6
1901,nausea,pregnancy,RELATED_TO,6
1957,numbness,diabetes,RELATED_TO,8
2016,pain,abscess,RELATED_TO,6


In [21]:
#filter weak relationships

filtered_relationships=relationship_counts[
    relationship_counts["count"]>=3
]

print(filtered_relationships.shape)

display(
    filtered_relationships.head(20)
)

(124, 4)


,source,target,relationship,count
18,abdominal pain,gallstones,RELATED_TO,4
32,abscess,antibiotics,TREATED_BY,4
36,abscess,painkillers,TREATED_BY,4
128,anemia,supplements,TREATED_BY,4
136,anxiety,anxiety,RELATED_TO,4
137,anxiety,anxiolytic,TREATED_BY,4
155,anxiety,hyperthyroidism,RELATED_TO,4
288,bleeding,endometriosis,RELATED_TO,4
293,bleeding,periods,RELATED_TO,4
318,bloating,gastritis,RELATED_TO,4


In [22]:
#sort by strongest relationships

filtered_relationships=filtered_relationships.sort_values(
    by="count",
    ascending=False
)

display(
    filtered_relationships.head(30)
)

,source,target,relationship,count
2142,pain,infection,RELATED_TO,16
622,cough,bronchitis,RELATED_TO,14
2093,pain,diabetes,RELATED_TO,10
614,cough,asthma,RELATED_TO,10
1413,infection,antibiotics,TREATED_BY,10
2236,pain,sciatica,RELATED_TO,10
1957,numbness,diabetes,RELATED_TO,8
2028,pain,appendicitis,RELATED_TO,8
2204,pain,pain,RELATED_TO,6
2278,pain,urinary tract infection,RELATED_TO,6


In [23]:
#save filtered relationships

filtered_relationships.to_csv(
    r"C:\Users\arsha\OneDrive\Desktop\Healthcare Retrieval using qql\Data\filtered_relationships.csv",
    index=False
)

print("filtered relationships saved")

filtered relationships saved


In [24]:
#top medical relationships

display(
    filtered_relationships.head(50)
)

,source,target,relationship,count
2142,pain,infection,RELATED_TO,16
622,cough,bronchitis,RELATED_TO,14
2093,pain,diabetes,RELATED_TO,10
614,cough,asthma,RELATED_TO,10
1413,infection,antibiotics,TREATED_BY,10
2236,pain,sciatica,RELATED_TO,10
1957,numbness,diabetes,RELATED_TO,8
2028,pain,appendicitis,RELATED_TO,8
2204,pain,pain,RELATED_TO,6
2278,pain,urinary tract infection,RELATED_TO,6


In [25]:
#check dizziness relationships

dizziness_relations=filtered_relationships[
    filtered_relationships["source"]=="dizziness"
]

display(dizziness_relations)

,source,target,relationship,count
827,dizziness,hypoglycemia,RELATED_TO,4
818,dizziness,cardiac disease,RELATED_TO,4


In [26]:
#relationship statistics

print("total relationships")

print(
    len(relationship_df)
)
print("\nfiltered relationships")
print(
    len(filtered_relationships)
)
print("\nremoved noisy relationships")
print(
    len(relationship_df)-len(filtered_relationships)
)

total relationships
7300

filtered relationships
124

removed noisy relationships
7176


In [27]:
#strongest symptom relationships

symptom_relations=filtered_relationships[
    filtered_relationships["relationship"]=="RELATED_TO"
]

display(
    symptom_relations.head(25)
)

,source,target,relationship,count
2142,pain,infection,RELATED_TO,16
622,cough,bronchitis,RELATED_TO,14
2093,pain,diabetes,RELATED_TO,10
614,cough,asthma,RELATED_TO,10
2236,pain,sciatica,RELATED_TO,10
1957,numbness,diabetes,RELATED_TO,8
2028,pain,appendicitis,RELATED_TO,8
2204,pain,pain,RELATED_TO,6
2278,pain,urinary tract infection,RELATED_TO,6
2259,pain,tooth infection,RELATED_TO,6


In [28]:
#saving final graph-ready dataset

filtered_relationships.to_csv(
    r"C:\Users\arsha\OneDrive\Desktop\Healthcare Retrieval using qql\Data\graph_ready_relationships.csv",
    index=False
)

print("graph ready relationships saved successfully")

graph ready relationships saved successfully
